# Caderno 8 - Comparação de ranking de sistemas de busca gerado por diversos qrels

## 1. Parâmetros

In [1]:
PASTA_DOCS_QRELS = './dados/outputs/0 - qrel - docs - query - raw_human_eval/'
PASTA_QRELS_LLMS = './dados/outputs/4 - qrel llm/'
PASTA_RUNS = './dados/outputs/7 - runs/'

ARQUIVO_QRELS = f'{PASTA_DOCS_QRELS}qrel.csv'
ARQUIVO_RAW_HUMAN_EVAL = f'{PASTA_DOCS_QRELS}raw_human_eval.csv'

In [2]:
MAPA_NOME_RUNS = {
    'run_bm25_0.9_0.4_assunto_e_texto': 'S_1',
    'run_bm25_0.9_0.4_texto': 'S_2',
    'run_bm25_0.82_0.68_assunto_e_texto': 'S_3',
    'run_bm25_0.82_0.68_texto': 'S_4',
    'run_bm25_1_0.3_assunto_e_texto': 'S_5',
    'run_bm25_1_0.3_texto': 'S_6',
    'run_bm25_1_0.9_assunto_e_texto': 'S_7',
    'run_bm25_1_0.9_texto': 'S_8',
    'run_text-embedding-3-large': 'S_9',
    'run_text-embedding-3-small': 'S_10',
    'run_gemini-embedding-001': 'S_11',
    'run_qwen_4b': 'S_12',
    'run_qwen_8b': 'S_13',
    'run_bm25_1_0.9_texto_text-embedding-3-small': 'S_14',
    'run_bm25_1_0.9_texto_gemini-embedding-001': 'S_15'
}

MAPA_NOME_QRELS = {
    'qrel_deepseek-chat_cot': 'deepseek_cot',
    'qrel_deepseek-chat_simple': 'deepseek_simple',
    'qrel_gpt-5-mini-2025-08-07_cot': 'gpt-5-mini_cot',
    'qrel_gpt-5-mini-2025-08-07_simple': 'gpt-5-mini_simple',
    'qrel_sabiazinho-4-2026-01-06_cot': 'sabiazinho-4_cot',
    'qrel_sabiazinho-4-2026-01-06_simple': 'sabiazinho-4_simple'
}

# Guarda a ordem para exibição dos gráficos
NOME_QRELS_REF = "Reference"
subtitle_order = [
    NOME_QRELS_REF,
    "A_1",
    "A_2",
    "A_3",
    "A_4",
    "deepseek_simple",
    "gpt-5-mini_simple",
    "sabiazinho-4_simple",
    "deepseek_cot",
    "gpt-5-mini_cot",
    "sabiazinho-4_cot",
]

## 2. Carrega todos os qrels

In [3]:
# Carrega todos os qrels de LLMs
from pathlib import Path
import pandas as pd

qrels = {}

# Carrega o qrels humano
qrels['Reference'] = pd.read_csv(ARQUIVO_QRELS)

########## CARREGA QRELS LLM ########## 

# Percorre todos os arquivos .csv da pasta
PASTA_QRELS_LLMS = Path(PASTA_QRELS_LLMS)
for arquivo in PASTA_QRELS_LLMS.glob("*.csv"):
    # Nome do arquivo sem extensão
    nome_qrels = arquivo.stem
    
    # Lê o CSV
    df = pd.read_csv(arquivo)
    
    # Salva no dicionário
    qrels[MAPA_NOME_QRELS.get(nome_qrels, nome_qrels)] = df


########## CARREGA QRELS ANOTADORES HUMANOS INDIVIDUAIS ########## 

# Gera os qrels de cada anotador individual
df_raw_human_eval = pd.read_csv(ARQUIVO_RAW_HUMAN_EVAL)
# Renomeia SCORE
df_raw_human_eval = df_raw_human_eval.rename(columns={"AVALIACAO": "SCORE"})
# Agrupa por usuário/query_key/score
df_raw_human_eval = df_raw_human_eval.sort_values(
    by=["USUARIO_AVALIADOR", "QUERY_KEY", "SCORE"],
    ascending=[True, True, False]
)
# Cria RANK (reinicia para cada QUERY_KEY dentro de cada avaliador)
df_raw_human_eval["RANK"] = (
    df_raw_human_eval.groupby(["USUARIO_AVALIADOR", "QUERY_KEY"])
      .cumcount() + 1
)
# Cria um df para cada avaliador
for user, grupo in df_raw_human_eval.groupby("USUARIO_AVALIADOR"):
    qrels[user] = grupo[["QUERY_KEY", "DOC_KEY", "SCORE", "RANK"]].reset_index(drop=True)

print(f"{len(qrels)} arquivos carregados.")

11 arquivos carregados.


## 3. Carrega todos os runs

In [4]:
runs = {}

# Percorre todos os arquivos .csv da pasta
PASTA_RUNS = Path(PASTA_RUNS)
for arquivo in PASTA_RUNS.glob("*.csv"):
    # Nome do arquivo sem extensão
    nome_run = arquivo.stem
    
    # Lê o CSV
    df = pd.read_csv(arquivo)
    
    # Salva no dicionário
    runs[MAPA_NOME_RUNS[nome_run]] = df

print(f"{len(runs)} arquivos carregados.")

15 arquivos carregados.


In [5]:
# Exporta todos os runs em um único df, caso seja necessário.
import pandas as pd

dfs = []

for modelo, df in runs.items():
    df_copy = df.copy()
    df_copy["MODELO"] = modelo
    dfs.append(df_copy)

# Concatena tudo
df_final = pd.concat(dfs, ignore_index=True)

# Opcional: colocar MODELO como primeira coluna
cols = ["MODELO"] + [c for c in df_final.columns if c != "MODELO"]
df_final = df_final[cols]

# Salva
#df_final.to_csv("df_todos_runs.csv", index=False)

#print("CSV gerado com sucesso.")

## 4. Gera métricas para todos os runs usando todos os pares de qrels

In [6]:
from metricas import metricas

results = []

for qrels_name, df_qrels in qrels.items():
    for run_name, df_run in runs.items():
        
        # Aqui você calcula as métricas
        df_resultados = metricas(df_run, df_qrels, aproximacao_trec_eval=True, k=[5, 10])

        metricas_disponiveis = ['nDCG@5', 'P@5', 'R@5', 'MRR@5', 'nDCG@10', 'P@10', 'R@10', 'MRR@10']
        for metrica_name in metricas_disponiveis:
            results.append({
                "qrels": qrels_name,
                "run": run_name,
                "metric": metrica_name,
                "value": df_resultados[metrica_name].mean(),
                "std": df_resultados[metrica_name].std()
            })

df_metricas = pd.DataFrame(results)

In [7]:
x = df_metricas[df_metricas.qrels==NOME_QRELS_REF]
x = x[x.metric == 'nDCG@10'].sort_values('value', ascending=False)
#x = x[x.run == 'run_bm25_0.82_0.68_texto']
x

,qrels,run,metric,value,std


## 5. Calcula a correlação do ranking dos sistemas usando Kendall Tau e coeficiente de Spearman

In [8]:
from scipy.stats import kendalltau, spearmanr
import pandas as pd

resultados_correlacao = []

metricas_disponiveis = df_metricas["metric"].unique()

for metrica in metricas_disponiveis:
    # Filtra apenas essa métrica
    df_metrica = df_metricas[df_metricas["metric"] == metrica]
    
    # Pivot: linhas = runs, colunas = qrels
    pivot = df_metrica.pivot(
        index="run",
        columns="qrels",
        values="value"
    )
       
    # Ranking induzido pelo qrels humano
    valores_human = pivot[NOME_QRELS_REF]
    
    for qrels_name in pivot.columns:
        if qrels_name == NOME_QRELS_REF:
            continue
        
        valores_outro = pivot[qrels_name]
        
        # Calcula Kendall tau-b
        tau, p_tau = kendalltau(valores_human, valores_outro)
        
        # Calcula Spearman rho
        rho, p_rho = spearmanr(valores_human, valores_outro)
        
        resultados_correlacao.append({
            "metric": metrica,
            "qrels_comparado": qrels_name,
            "kendall_tau": tau,
            "kendall_pvalue": p_tau,
            "spearman_rho": rho,
            "spearman_pvalue": p_rho
        })

df_correlacoes = pd.DataFrame(resultados_correlacao)

# Ordena pelo nome do qrels
df_correlacoes['qrels_comparado'] = pd.Categorical(
    df_correlacoes['qrels_comparado'],
    categories=subtitle_order,
    ordered=True
)
df_correlacoes = df_correlacoes.sort_values('qrels_comparado')


#df_correlacoes.sort_values(["metric", "kendall_tau"], ascending=[True, False], inplace=True)

df_correlacoes.reset_index(drop=True, inplace=True)

df_correlacoes

KeyError: 'human'

In [ ]:
metricas_desejadas = ["P@10", "R@10", "MRR@10", "nDCG@10"]

# Filtra apenas as métricas desejadas
df_filtrado = df_correlacoes[
    df_correlacoes["metric"].isin(metricas_desejadas)
].copy()

# Garante a ordem correta
df_filtrado["metric"] = pd.Categorical(
    df_filtrado["metric"],
    categories=metricas_desejadas,
    ordered=True
)

df_filtrado = df_filtrado.sort_values("metric")

# Impressão organizada
for metrica in metricas_desejadas:
    df_metrica = df_filtrado[df_filtrado["metric"] == metrica]
    
    print(f"\n=== {metrica} ===")
    
    for _, row in df_metrica.iterrows():
        print(
            f"{row['qrels_comparado']}: "
            f"Kendall = {row['kendall_tau']:.4f} | "
            f"Spearman = {row['spearman_rho']:.4f}"
        )

In [ ]:
metricas_desejadas = ["P@10", "R@10", "MRR@10", "nDCG@10"]

# Filtra métricas desejadas
df_filtrado = df_correlacoes[
    df_correlacoes["metric"].isin(metricas_desejadas)
].copy()

# Garante a ordem correta das métricas
df_filtrado["metric"] = pd.Categorical(
    df_filtrado["metric"],
    categories=metricas_desejadas,
    ordered=True
)

# Pivot
df_expandido = (
    df_filtrado
    .pivot(
        index="qrels_comparado",
        columns="metric",
        values=["kendall_tau", "spearman_rho"]
    )
)

# Reorganiza explicitamente as colunas na ordem desejada
colunas_ordenadas = []
for metrica in metricas_desejadas:
    colunas_ordenadas.append(("kendall_tau", metrica))
    colunas_ordenadas.append(("spearman_rho", metrica))

df_expandido = df_expandido[colunas_ordenadas]

# Achata o multi-index
df_expandido.columns = [
    f"{corr}_{metrica}" for corr, metrica in df_expandido.columns
]

df_expandido = df_expandido.reset_index()

# Exibição com 2 casas decimais
df_expandido = df_expandido.round(2)

df_expandido

In [ ]:
import math
import matplotlib.pyplot as plt

metricas_disponiveis = df_metricas["metric"].unique()

for metrica in metricas_disponiveis:
    df_metrica = df_metricas[df_metricas["metric"] == metrica]
    
    # Pivot para facilitar acesso
    pivot = df_metrica.pivot(
        index="run",
        columns="qrels",
        values="value"
    )
       
    # Ordena runs pelo desempenho humano (decrescente)
    pivot = pivot.sort_values(NOME_QRELS_REF, ascending=False)
    
    runs_ordenados = pivot.index.tolist()
    x = range(len(runs_ordenados))
    
    plt.figure(figsize=(10, 6))
    
    # Dicionário para armazenar linhas (handle da legenda)
    lines_dict = {}
    
    # ===== Plot humano =====
    line_human, = plt.plot(
        x,
        pivot[NOME_QRELS_REF],
        marker="o",
        linewidth=3,
        label=NOME_QRELS_REF
    )
    lines_dict[NOME_QRELS_REF] = line_human
    
    # ===== Plot outros qrels =====
    for qrels_name in pivot.columns:
        if qrels_name == NOME_QRELS_REF:
            continue
        
        line, = plt.plot(
            x,
            pivot[qrels_name],
            marker="o",
            linestyle="--",
            alpha=0.8,
            label=qrels_name
        )
        lines_dict[qrels_name] = line
    
    # =========================
    # 🔹 Ordena legenda segundo subtitle_order
    # =========================
    ordered_handles = [
        lines_dict[name]
        for name in subtitle_order
        if name in lines_dict
    ]
    
    # Configurações do gráfico
    plt.xticks(x, runs_ordenados, rotation=90)
    plt.xlabel("Runs (ordered by human ranking)")
    plt.ylabel(metrica)
    plt.title(f"{metrica} - Ranking Stability Across Qrels")
    
    # 🔹 Legenda abaixo do gráfico
    plt.legend(
        handles=ordered_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        ncol=math.ceil(len(ordered_handles)/3)
    )
    
    plt.tight_layout()
    #plt.savefig(f"ranking_across_qrels_{metrica.replace('@', '_')}.png", dpi=300, bbox_inches="tight")
    plt.show()